# Pipeline Output Analysis & Visualization (Extended)

This notebook loads the classified events produced by the pipeline
(`classified_packets.csv`) and analyzes them.

In addition to the required time-series chart, I added a few of the
**suggested experiments** from the lab:
- distribution by MITRE **tactic** and by **technique**
- breakdown **per host** and **per user**
- readable MITRE labels instead of just the codes


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CSV_PATH = Path("classified_packets.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(
        "classified_packets.csv was not found. Run 1. Producer.ipynb and "
        "2. Consumer_Classifier.ipynb first, then rerun this notebook."
    )

df = pd.read_csv(CSV_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

print("Total classified events:", len(df))
print("Time range:", df["timestamp"].min(), "->", df["timestamp"].max())
df.head()


In [ ]:
# Readable names for the MITRE codes used by the classifier
TACTIC_NAMES = {
    "TA0001": "Initial Access",
    "TA0002": "Execution",
    "TA0006": "Credential Access",
}
TECHNIQUE_NAMES = {
    "T1059":     "Command & Scripting Interpreter",
    "T1059.001": "PowerShell",
    "T1059.003": "Windows Command Shell",
    "T1078":     "Valid Accounts",
    "T1110":     "Brute Force",
}

df["tactic_name"] = df["mitre_tactic"].map(TACTIC_NAMES)
df["technique_name"] = df["mitre_technique"].map(TECHNIQUE_NAMES)


## 1. Required chart: events over time by tactic

In [ ]:
# Group events into 30-minute time buckets, stacked by tactic.
# This produces a readable report-level view and matches the lab report caption.
df["time_bucket"] = df["timestamp"].dt.floor("30min")

pivot = (
    df.groupby(["time_bucket", "mitre_tactic"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

# Reindex onto a continuous 30-minute range between the first and last event so
# inactive periods appear as gaps (empty buckets) instead of being collapsed
# together. This makes the chart match Figure 1 in the lab report.
full_range = pd.date_range(pivot.index.min(), pivot.index.max(), freq="30min")
pivot = pivot.reindex(full_range, fill_value=0)

fig, ax = plt.subplots(figsize=(18, 8))
pivot.plot(kind="bar", stacked=True, width=0.8, ax=ax)

ax.set_title("Classified Security Events Over Time by MITRE ATT&CK Tactic (30-minute windows)")
ax.set_xlabel("Time bucket (UTC)")
ax.set_ylabel("Number of events")
ax.legend(title="MITRE Tactic", bbox_to_anchor=(1.05, 1), loc="upper left")

# Keep the x-axis readable when many buckets exist.
ax.set_xticklabels([str(label.get_text())[:16] for label in ax.get_xticklabels()], rotation=45, ha="right")

plt.tight_layout()
plt.show()


### Interpretation of the time-series chart

The gaps between active time buckets indicate that the producer was started and stopped across several sessions rather than running continuously. This is expected for a lab execution. In a real SOC pipeline, the same visualization would help identify bursty event periods, long inactivity gaps, or abnormal spikes in a specific MITRE tactic.


## 2. Distribution by tactic and by technique

In [ ]:
tactic_counts = df["tactic_name"].value_counts()
technique_counts = df["technique_name"].value_counts()

print("Events per tactic:")
print(tactic_counts, "\n")
print("Events per technique:")
print(technique_counts)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
tactic_counts.plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Events per MITRE ATT&CK Tactic")
axes[0].set_ylabel("Number of events")
axes[0].tick_params(axis="x", rotation=20)

technique_counts.plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Events per MITRE ATT&CK Technique")
axes[1].set_xlabel("Number of events")

plt.tight_layout()
plt.show()


## 3. Extra experiment: breakdown per host and per user

In [ ]:
host_pivot = df.groupby(["host", "tactic_name"]).size().unstack(fill_value=0)
user_pivot = df.groupby(["user", "tactic_name"]).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

host_pivot.plot(kind="bar", stacked=True, ax=axes[0])
axes[0].set_title("Tactic composition per host")
axes[0].set_ylabel("Number of events")
axes[0].tick_params(axis="x", rotation=0)

user_pivot.plot(kind="bar", stacked=True, ax=axes[1])
axes[1].set_title("Tactic composition per user")
axes[1].set_ylabel("Number of events")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


## 4. Interpretation and conclusion

The event distribution is consistent with the generator logic. Since the producer randomly chooses between `process_start` and `user_login`, **Execution (TA0002)** is expected to represent approximately half of the dataset. The remaining events are split between successful and failed logins, which are mapped to **Initial Access (TA0001)** and **Credential Access (TA0006)**.

The host and user breakdowns do not show a dominant source, which is expected for synthetic uniformly generated data. In a real SOC environment, a strong concentration of failed logins on one user or one host would be a potential anomaly requiring investigation.

This notebook therefore validates both parts of the lab: the pipeline successfully converts raw synthetic Windows events into MITRE ATT&CK-labelled records, and the resulting CSV can be used for offline security analysis.
